In [21]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8')

# Load Database
DB_PATH = '../models/expert_system_db.pkl'
with open(DB_PATH, 'rb') as f:
    crops_db = pickle.load(f)

print(f"✅ Loaded {len(crops_db)} crops from Expert System Database!")

def predict_live_crops(sensor_data, top_n=3):
    # Temporary dict to hold ALL raw scores before scaling
    temp_all_results = {'Tree': [], 'Plant': [], 'Dry Fruit': []}
    final_results = {'Tree': [], 'Plant': [], 'Dry Fruit': []}
    
    # STEP 1: CALCULATE RAW ABSOLUTE SCORES
    for crop in crops_db:
        score, total = 0, 0
        
        temp = sensor_data.get('temp', 30)
        if crop['temp_min'] <= temp <= crop['temp_max']: score += 25
        elif abs(temp - crop['temp_min']) <= 5 or abs(temp - crop['temp_max']) <= 5: score += 12
        total += 25

        humidity = sensor_data.get('humidity', 50)
        if crop['humidity_min'] <= humidity <= crop['humidity_max']: score += 20
        elif abs(humidity - crop['humidity_min']) <= 10 or abs(humidity - crop['humidity_max']) <= 10: score += 10
        total += 20

        soil = sensor_data.get('soil_moisture', 40)
        if crop['soil_moisture_min'] <= soil <= crop['soil_moisture_max']: score += 20
        elif abs(soil - crop['soil_moisture_min']) <= 10 or abs(soil - crop['soil_moisture_max']) <= 10: score += 10
        total += 20

        ph = sensor_data.get('ph', 6.5)
        if crop['ph_min'] <= ph <= crop['ph_max']: score += 15
        elif abs(ph - crop['ph_min']) <= 0.5 or abs(ph - crop['ph_max']) <= 0.5: score += 7
        total += 15

        light = sensor_data.get('light', 70)
        if crop['light_min'] <= light <= crop['light_max']: score += 10
        total += 10

        rain = sensor_data.get('rain_annual_mm', 500)
        if crop['rain_min'] <= rain <= crop['rain_max']: score += 10
        total += 10

        score += crop.get('priority', 5) * 0.5
        
        # Raw percentage
        raw_pct = (score / (total + 5)) * 100

        crop_info = {
            'name': crop['name'],
            'raw_pct': raw_pct, # Storing for math
            'grow_days': crop['grow_days'],
            'water_need': crop['water_need'],
            'category': crop.get('category', 'Plant')
        }
        
        cat = crop_info['category']
        if cat in temp_all_results:
            temp_all_results[cat].append(crop_info)

    # STEP 2: APPLY RELATIVE NORMALIZATION (The UI/UX Fix)
    for cat, crops in temp_all_results.items():
        if not crops: continue
        
        # Find the top raw score in this specific category
        max_raw_score = max(c['raw_pct'] for c in crops)
        
        for c in crops:
            # Scale top performer to 99.0%, and rest relative to it
            if max_raw_score > 0:
                relative_pct = (c['raw_pct'] / max_raw_score) * 99.0
            else:
                relative_pct = 0
                
            c['match_pct'] = round(relative_pct, 1)

        # Sort by final match percentage
        crops.sort(key=lambda x: x['match_pct'], reverse=True)
        # Pick Top N
        final_results[cat] = crops[:top_n]

    return final_results

✅ Loaded 250 crops from Expert System Database!


In [24]:
# Real Live Sensor Data from ESP32
live_sensor_reading = {
    'temp': 29.7,           
    'humidity': 85.2,       
    'soil_moisture': 100,   
    'ph': 6.8,              
    'light': 57,            
    'rain_annual_mm': 600   
}

# YAHAN CHANGE HAI: Naya function call kar rahe hain purane model ki jagah
results = predict_live_crops(live_sensor_reading, top_n=3)

print("="*60)
print(" 🌾 LIVE SENSOR CROP RECOMMENDATION SYSTEM (SCALED) 🌾")
print("="*60)
print(f"📊 ESP32 Readings: Temp={live_sensor_reading['temp']}°C | "
      f"Humidity={live_sensor_reading['humidity']}% | "
      f"Soil={live_sensor_reading['soil_moisture']}% | "
      f"pH={live_sensor_reading['ph']}")
print("="*60)

for category, crops in results.items():
    icon = '🌳' if category == 'Tree' else '🌿' if category == 'Plant' else '🥜'
    print(f"\n{icon} TOP RECOMMENDATIONS FOR {category.upper()}S:")
    for i, crop in enumerate(crops, 1):
        print(f"  {i}. {crop['name']:25s} | Match: {crop['match_pct']:5.1f}% | "
              f"Grow Time: {crop['grow_days']:4d} days | Water Need: {crop['water_need']}")

 🌾 LIVE SENSOR CROP RECOMMENDATION SYSTEM (SCALED) 🌾
📊 ESP32 Readings: Temp=29.7°C | Humidity=85.2% | Soil=100% | pH=6.8

🌳 TOP RECOMMENDATIONS FOR TREES:
  1. Oud (Agarwood)            | Match:  99.0% | Grow Time: 3650 days | Water Need: High
  2. Cacao (Chocolate)         | Match:  96.4% | Grow Time: 1825 days | Water Need: High
  3. Vanilla (Vine Tree)       | Match:  96.4% | Grow Time: 1095 days | Water Need: High

🌿 TOP RECOMMENDATIONS FOR PLANTS:
  1. Rice                      | Match:  99.0% | Grow Time:  150 days | Water Need: High
  2. Jute                      | Match:  98.3% | Grow Time:  120 days | Water Need: High
  3. Tomato                    | Match:  85.8% | Grow Time:   90 days | Water Need: Medium

🥜 TOP RECOMMENDATIONS FOR DRY FRUITS:
  1. Fox Nut (Makhana)         | Match:  99.0% | Grow Time:  150 days | Water Need: High
  2. Kola Nut                  | Match:  85.6% | Grow Time: 2555 days | Water Need: High
  3. Dried Coconut (Copra)     | Match:  75.1% | Grow Tim